# 🎯 User-Job Matching — Silver → `user_job_matches`

**Purpose**: Match every active user against every active silver job.
- Skill overlap scoring (60% weight)
- Experience fit scoring (40% weight)
- Skips already-matched or applied jobs
- Near-matches also included (threshold: 40+)
- Parallel execution with ThreadPoolExecutor

**Threshold**: match_score >= 40 → included (user can see and decide)

In [0]:
import uuid, json
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit, current_timestamp

spark = SparkSession.builder.getOrCreate()

CATALOG         = "jobs_automation_db"
SILVER_TABLE    = f"{CATALOG}.default.jobs_clean_silver"
MATCHES_TABLE   = f"{CATALOG}.default.user_job_matches"
USERS_TABLE     = f"{CATALOG}.users_schema.users"
RESUMES_TABLE   = f"{CATALOG}.users_schema.user_resumes"
CLIPBOARD_TABLE = f"{CATALOG}.users_schema.user_application_clipboard"
SUBMISSIONS_TBL = f"{CATALOG}.default.job_submissions"

MIN_MATCH_SCORE   = 40    # Include near-matches too
PARALLEL_WORKERS  = 10    # Parallel user matching

print(f"✅ Config ready. Matching threshold: {MIN_MATCH_SCORE}")

In [0]:
# Load all active users
users_list = spark.sql(f"""
    SELECT u.user_id, u.full_name, u.total_experience_years, u.min_match_score,
           r.skills_extracted, r.years_experience, r.resume_id,
           r.work_history_json
    FROM {USERS_TABLE} u
    LEFT JOIN {RESUMES_TABLE} r
        ON u.user_id = r.user_id AND r.is_primary = true
    WHERE u.is_active = true
""").collect()

print(f"👥 Active users: {len(users_list)}")

# Load active silver jobs
jobs_list = spark.sql(f"""
    SELECT job_hash, job_title, company_name, location, remote_type,
           tech_stack, experience_min, experience_max,
           salary_range, apply_link, easy_apply_link, hr_email,
           validation_score, ai_summary, portal
    FROM {SILVER_TABLE}
    WHERE active = true
    AND validation_status != 'Junk'
""").collect()

print(f"💼 Active silver jobs: {len(jobs_list)}")

# Load already-matched pairs (user_id, job_hash)
try:
    existing_matches = set(
        spark.sql(f"SELECT user_id || '||' || job_hash FROM {MATCHES_TABLE}")
        .rdd.flatMap(lambda x: x).collect()
    )
except Exception:
    existing_matches = set()

# Load already-submitted pairs
try:
    existing_subs = set(
        spark.sql(f"SELECT user_id || '||' || job_hash FROM {SUBMISSIONS_TBL}")
        .rdd.flatMap(lambda x: x).collect()
    )
except Exception:
    existing_subs = set()

print(f"🔄 Existing matches: {len(existing_matches)} | Submissions: {len(existing_subs)}")

In [0]:
def normalize_skills(skills_str: str) -> set:
    """Normalize comma-separated skills to lowercase set."""
    if not skills_str:
        return set()
    return {s.strip().lower() for s in skills_str.split(',') if s.strip()}


def score_experience_fit(user_exp: float, job_min: int, job_max: int) -> tuple:
    """
    Returns (score 0-40, fit_label).
    'exact' = full 40 pts
    'over'  = 30 pts (overqualified — still valid)
    'under' = 20 pts if within 2 yrs, 10 pts if 3-4 yrs under, 0 if 5+ yrs under
    """
    if user_exp is None:
        return (20, "unknown")
    
    gap = user_exp - job_min  # positive = over, negative = under
    
    if -2 <= gap <= 99:  # Within 2 years below minimum OR over
        if gap >= 0:
            label = "exact" if gap <= 3 else "over"
            return (40 if label == "exact" else 30, label)
        else:  # slightly under but close
            return (25, "near")
    elif gap >= -4:
        return (15, "under")
    else:
        return (5, "under")


def match_user_to_job(user, job) -> dict | None:
    """Score one user against one job. Returns match dict or None."""
    pair_key = f"{user.user_id}||{job.job_hash}"
    
    # Skip already matched or submitted
    if pair_key in existing_matches or pair_key in existing_subs:
        return None

    user_skills = normalize_skills(user.skills_extracted or "")
    job_skills  = normalize_skills(job.tech_stack or "")

    # Skill overlap
    overlap = user_skills & job_skills
    gap     = job_skills - user_skills

    if job_skills:
        skill_pct   = round(len(overlap) / len(job_skills) * 100)
        skill_score = min(int(skill_pct * 0.6), 60)  # max 60 pts
    else:
        skill_pct   = 50
        skill_score = 30

    # Experience fit
    user_exp = float(user.total_experience_years or user.years_experience or 0)
    exp_score, exp_fit = score_experience_fit(user_exp, job.experience_min or 0, job.experience_max or 99)

    total_score = skill_score + exp_score

    # Respect user's min_match_score preference
    user_min = int(user.min_match_score or 40)
    if total_score < max(MIN_MATCH_SCORE, user_min):
        return None

    return {
        "match_id":         str(uuid.uuid4()),
        "user_id":          user.user_id,
        "job_hash":         job.job_hash,
        "clipboard_id":     None,   # Will be set when user picks a clipboard
        "match_score":      total_score,
        "skill_match_pct":  skill_pct,
        "experience_fit":   exp_fit,
        "skill_overlap":    ", ".join(sorted(overlap)),
        "skill_gap":        ", ".join(sorted(gap)),
        "match_reasons_json": json.dumps({
            "skill_score": skill_score,
            "exp_score": exp_score,
            "skill_overlap_count": len(overlap),
            "job_skill_count": len(job_skills),
        }),
        "matched_at":       datetime.now(),
        "resume_generated": False,
        "resume_id":        None,
        "resume_generated_at": None,
        "status":           "matched",
    }

print("✅ Matching functions defined")

In [0]:
all_matches = []

def process_user(user):
    user_matches = []
    for job in jobs_list:
        result = match_user_to_job(user, job)
        if result:
            user_matches.append(result)
    return user.user_id, user_matches

# Parallel matching across users
with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as executor:
    futures = {executor.submit(process_user, user): user for user in users_list}
    for future in as_completed(futures):
        try:
            user_id, matches = future.result(timeout=120)
            all_matches.extend(matches)
            print(f"  ✅ User {user_id[:8]}...: {len(matches)} new matches")
        except Exception as e:
            print(f"  ❌ Matching error: {e}")

print(f"\n🎯 Total new matches: {len(all_matches)}")

In [0]:
if all_matches:
    MATCH_SCHEMA = StructType([
        StructField("match_id",           StringType(),    False),
        StructField("user_id",            StringType(),    True),
        StructField("job_hash",           StringType(),    True),
        StructField("clipboard_id",       StringType(),    True),
        StructField("match_score",        IntegerType(),   True),
        StructField("skill_match_pct",    IntegerType(),   True),
        StructField("experience_fit",     StringType(),    True),
        StructField("skill_overlap",      StringType(),    True),
        StructField("skill_gap",          StringType(),    True),
        StructField("match_reasons_json", StringType(),    True),
        StructField("matched_at",         TimestampType(), True),
        StructField("resume_generated",   BooleanType(),   True),
        StructField("resume_id",          StringType(),    True),
        StructField("resume_generated_at",TimestampType(), True),
        StructField("status",             StringType(),    True),
    ])

    matches_df = spark.createDataFrame(all_matches, schema=MATCH_SCHEMA)
    matches_df.write.format("delta").mode("append").saveAsTable(MATCHES_TABLE)
    print(f"✅ Saved {len(all_matches)} new matches to {MATCHES_TABLE}")
else:
    print("ℹ️  No new matches found.")

In [0]:
%sql
SELECT 
    m.user_id,
    u.full_name,
    COUNT(*) AS total_matches,
    AVG(m.match_score) AS avg_score,
    SUM(CASE WHEN m.resume_generated THEN 1 ELSE 0 END) AS resumes_done,
    SUM(CASE WHEN m.status = 'applied' THEN 1 ELSE 0 END) AS applied
FROM jobs_automation_db.default.user_job_matches m
JOIN jobs_automation_db.users_schema.users u ON m.user_id = u.user_id
GROUP BY m.user_id, u.full_name
ORDER BY total_matches DESC